# Black-Scholes Option Pricing and Put-Call Parity

In this notebook, I calculate the theoretical prices of a call and put option using the Black-Scholes model. At the end, I use put-call parity to check whether the prices are consistent.

In [1]:
%pip install py_vollib

Note: you may need to restart the kernel to use updated packages.


### Import the libraries

I import the libraries needed for the calculations. `norm` is used for the cumulative normal distribution, `bs` lets me compare my result with the `py_vollib` library, and `numpy` is used for mathematical calculations.

In [2]:
from scipy.stats import norm
from py_vollib.black_scholes import black_scholes as bs
import numpy as np

/var/folders/lm/xrsljtd570b50x27zd0qmbz00000gn/T/ipykernel_2281/3828731820.py:2: DeprecationWarning: py_vollib is deprecated and will be removed in a future release; please import from vollib instead.
  from py_vollib.black_scholes import black_scholes as bs


### Set the input values

I enter the current stock price (`S`), strike price (`K`), annual risk-free rate (`r`), time to expiry in years (`T`), and annual volatility (`sigma`). These are the inputs required by the Black-Scholes model.

In [3]:
S= 22698.85
K=21000
r= 0.0573
T= 5.9930555556/365
sigma= 0.4671

### Calculate d1

I calculate `d1`, which is one of the two key values used in the Black-Scholes option pricing formula.

In [4]:
d1 = (np.log(S/K)+(r+ sigma**2/2)*T)/(sigma*np.sqrt(T))
print(d1)


1.345355260400638


### Calculate d2

I calculate `d2` by subtracting volatility multiplied by the square root of time from `d1`.

In [5]:
d2= d1 - (sigma*np.sqrt(T))
print(d2)

1.2855020311384775


### Calculate the call option price

I use the Black-Scholes call formula to calculate the theoretical price of the call option.

In [6]:
call = S*norm.cdf(d1,0,1)-norm.cdf(d2,0,1)*K*np.exp(-r*T)
print(call)

1776.1225255513964


### Calculate the put option price

I use the Black-Scholes put formula to calculate the theoretical price of the put option.

In [7]:
put = norm.cdf(-d2,0,1)*K*np.exp(-r*T)-S*norm.cdf(-d1,0,1)
print(put)


57.52443664473935


### Verify prices using `py_vollib`

I calculate the call and put prices again using `py_vollib`. I can compare these results with my own calculations to check whether the formulas are producing similar values.

In [8]:
call_price = bs("c",S,K,T,r,sigma)
print(call_price)

1776.1225255513943


In [9]:
put_price = bs("p",S,K,T,r,sigma)
print(put_price)

57.52443664473997


### Put-Call Parity: Check for a mismatch

For European call and put options with the same stock, strike price, and expiry, put-call parity is:

\[
C + Ke^{-rT} = P + S
\]

I compare both sides of this equation. If the difference is close to zero, the prices satisfy put-call parity. A mismatch may be worth investigating, but it does not by itself prove an arbitrage opportunity. For that, I need actual market prices and must consider bid-ask spreads, transaction costs, and execution risk.

In [10]:
# Inputs
S = 22698.85
K = 21000
r = 0.0573
T = 5.9930555556/365
sigma = 0.4671

# Calculate d1 and d2
d1 = (np.log(S/K)+(r+sigma**2/2)*T)/(sigma*np.sqrt(T))
d2 = d1 - sigma*np.sqrt(T)

# Calculate call and put prices
call = S*norm.cdf(d1)-K*np.exp(-r*T)*norm.cdf(d2)
put = K*np.exp(-r*T)*norm.cdf(-d2)-S*norm.cdf(-d1)

# Put-Call Parity
left_side = call + K*np.exp(-r*T)
right_side = put + S

difference = left_side - right_side

print("Call Price:", call)
print("Put Price:", put)
print("Call + Present Value of Strike:", left_side)
print("Put + Stock Price:", right_side)
print("Difference:", difference)

if abs(difference) < 0.01:
    print("No put-call parity mismatch")
else:
    print("Possible parity mismatch - check market prices and costs")

Call Price: 1776.1225255513964
Put Price: 57.5244366447389
Call + Present Value of Strike: 22756.37443664474
Put + Stock Price: 22756.37443664474
Difference: 0.0
No put-call parity mismatch
